# Save Final Gradient Boosting Model

This notebook trains the selected final model for predicting `migraine_occurrence` and saves both the trained model and the feature list for later use in the project.

## 1. Import required libraries

The following libraries are used for data handling, model training, model evaluation, and saving the trained model files.

In [ ]:
from pathlib import Path
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

## 2. Load the dataset

The dataset is loaded using the full file path so that the notebook runs reliably in this VS Code setup.

In [ ]:
data_path = Path('/Users/cioran/Documents/migraine-prediction-project/data/migraine_dataset.csv')
print('Loading dataset from:', data_path)
df = pd.read_csv(data_path)
df.head()

## 3. Convert the `date` column to datetime

The `date` column is converted to datetime format so that date-based features can be created.

In [ ]:
df['date'] = pd.to_datetime(df['date'])
print(df['date'].dtype)

## 4. Create the improved features

This step creates the date-based features and interaction features that were used in the model improvement stage.

In [ ]:
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['sleep_stress_interaction'] = df['sleep_hours'] * df['stress_level']
df['screen_stress_interaction'] = df['screen_time'] * df['stress_level']
df['hydration_stress_interaction'] = df['hydration_level'] * df['stress_level']

df[[
    'day_of_week',
    'month',
    'sleep_stress_interaction',
    'screen_stress_interaction',
    'hydration_stress_interaction'
]].head()

## 5. Define `X` and `y`

The feature matrix `X` contains the improved feature set, and `y` contains the target variable `migraine_occurrence`.

In [ ]:
feature_names = [
    'sleep_hours',
    'mood_level',
    'stress_level',
    'hydration_level',
    'screen_time',
    'day_of_week',
    'month',
    'sleep_stress_interaction',
    'screen_stress_interaction',
    'hydration_stress_interaction'
]

target = 'migraine_occurrence'

X = df[feature_names]
y = df[target]

print('Shape of X:', X.shape)
print('Shape of y:', y.shape)

## 6. Split the data into training and testing sets

The dataset is split into training and testing subsets so that the model can be evaluated before being retrained on the full dataset.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)

## 7. Train a Gradient Boosting model

Gradient Boosting is trained because it was the strongest overall model during the model improvement stage.

In [ ]:
gb_model = GradientBoostingClassifier(random_state=42)
gb_model.fit(X_train, y_train)

## 8. Evaluate the model

The model is evaluated on the test set using standard classification metrics.

In [ ]:
y_pred = gb_model.predict(X_test)
y_prob = gb_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print('Gradient Boosting Performance')
print('Accuracy:', accuracy)
print('Precision:', precision)
print('Recall:', recall)
print('F1-score:', f1)
print('ROC-AUC:', roc_auc)
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

## 9. Train the final model on the full dataset

After evaluation, the final Gradient Boosting model is trained again using the full dataset so that all available information is used before saving the model.

In [ ]:
final_model = GradientBoostingClassifier(random_state=42)
final_model.fit(X, y)

## 10. Save the final model

The trained final model is saved in the `models` folder using `joblib`.

In [ ]:
model_path = Path('/Users/cioran/Documents/migraine-prediction-project/models/migraine_gradient_boosting_model.joblib')
joblib.dump(final_model, model_path)
print('Saved model to:', model_path)

## 11. Save the feature list

The feature names are saved separately so that the backend can later reconstruct the expected input order for predictions.

In [ ]:
feature_path = Path('/Users/cioran/Documents/migraine-prediction-project/models/feature_names.joblib')
joblib.dump(feature_names, feature_path)
print('Saved feature names to:', feature_path)

## 12. Final note

The saved model and feature list will later be loaded by the backend API so that the web application can generate migraine occurrence predictions from user input.